# Урок 05 - Агентный RAG


## Настройка

В этой записной книжке демонстрируется шаблон Agentic RAG (Retrieval-Augmented Generation) с использованием Microsoft Agent Framework.

**Требования:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — конечная точка вашего сервиса Azure AI Search
- `AZURE_SEARCH_API_KEY` — ваш API-ключ Azure AI Search
- Развертывание Azure OpenAI, настроенное через переменные окружения
- Аутентификация через Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Что такое Agentic RAG?

Традиционный RAG следует фиксированному конвейеру: сначала извлечение документов, затем генерация ответа. **Agentic RAG** идет дальше, предоставляя агенту автономию в принятии решений о том, **когда** и **как** получать информацию.

С Agentic RAG агент может:
- **Решать**, нужно ли извлекать информацию перед ответом на вопрос
- **Выбирать**, к какому источнику данных или инструменту обратиться
- **Оценивать** полученные результаты и выполнять повторные извлечения, если первый запрос недостаточен
- **Объединять** информацию из нескольких шагов извлечения в связный ответ

Это делает агента более гибким и точным по сравнению со статическим конвейером "извлечение-затем-генерация".


## Создание инструмента поиска

В Agentic RAG внешние источники данных оформляются как **инструменты**, которые агент может вызывать по запросу. Это позволяет агенту рассматривать извлечение как просто ещё одно действие, которое он может выполнить, а не как обязательный шаг.

Ниже мы определим базу знаний по путешествиям и предоставим её в виде инструмента, который агент может вызвать для поиска информации о направлениях.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Создание агента RAG

Теперь мы создаём агента, которому дано указание **всегда получать информацию перед ответом**. Агент использует инструмент `search_travel_knowledge`, чтобы опираться в своих ответах на базу знаний, а не на собственные тренировочные данные.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Итеративный поиск — паттерн «создатель-проверяющий»

Главное преимущество Agentic RAG — **итеративный поиск**. Агент может выполнять несколько циклов поиска, чтобы проверить, уточнить или расширить свои первоначальные результаты — аналогично рабочему процессу «создатель-проверяющий»:

1. **Шаг создателя**: агент извлекает первоначальную информацию и составляет ответ.
2. **Шаг проверяющего**: агент выполняет дополнительные поиски для проверки деталей или заполнения пробелов.

Ниже агенту задан вопрос, требующий сравнения нескольких направлений, что побуждает его искать несколько раз.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Резюме

В этом уроке вы узнали, как создать систему **Agentic RAG** с использованием Microsoft Agent Framework:

- **Agentic RAG** позволяет агентам самостоятельно решать, когда выполнять поиск информации, делая поиск динамичным, а не фиксированным.
- **Инструменты в качестве источников данных**: Внешние базы знаний (например, Azure AI Search) оборачиваются в инструменты, которые агент может вызывать.
- **Итеративный поиск**: Шаблон maker-checker позволяет агенту выполнять несколько раундов поиска — искать, проверять и уточнять — прежде чем выдать окончательный ответ.

В производственной среде вы замените встроенную `TRAVEL_KNOWLEDGE_BASE` на реальный индекс Azure AI Search для обработки масштабного поиска документов о путешествиях.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
